In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")
LANDING = "/Volumes/workspace/bronze/landing"


def read_landing(folder: str):
    """Read every JSON file in one landing folder and keep lineage columns."""
    return (
        spark.read.option("multiLine", True).json(f"{LANDING}/{folder}/")
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_file_modified", F.col("_metadata.file_modification_time"))
    )

In [0]:
price_raw = (
    read_landing("price")
    .select("_source_file", "_file_modified", "unix_seconds",
            F.posexplode("price").alias("i", "price"))
    .select(F.expr("unix_seconds[i]").alias("unix_seconds"), "price",
            "_source_file", "_file_modified")
    .withColumn("_ingested_at", F.current_timestamp())
)

(price_raw.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.bronze.price_raw"))
display(spark.table("workspace.bronze.price_raw").limit(10))

In [0]:
power_raw = (
    read_landing("power")
    .select("_source_file", "_file_modified", "unix_seconds",
            F.explode("production_types").alias("pt"))
    .select("_source_file", "_file_modified", "unix_seconds",
            F.col("pt.name").alias("source"),
            F.posexplode("pt.data").alias("i", "mw"))
    .select("source", F.expr("unix_seconds[i]").alias("unix_seconds"), "mw",
            "_source_file", "_file_modified")
    .withColumn("_ingested_at", F.current_timestamp())
)

(power_raw.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.bronze.power_raw"))
display(spark.table("workspace.bronze.power_raw").groupBy("source").count())

In [0]:
weather_raw = (
    read_landing("weather")
    .select("_source_file", "_file_modified", "hourly.*")
    .select("_source_file", "_file_modified", "temperature_2m", "wind_speed_10m",
            "shortwave_radiation", F.posexplode("time").alias("i", "time"))
    .select("time",
            F.expr("temperature_2m[i]").alias("temperature_2m"),
            F.expr("wind_speed_10m[i]").alias("wind_speed_10m"),
            F.expr("shortwave_radiation[i]").alias("shortwave_radiation"),
            "_source_file", "_file_modified")
    .withColumn("_ingested_at", F.current_timestamp())
)

(weather_raw.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.bronze.weather_raw"))
display(spark.table("workspace.bronze.weather_raw").limit(10))